# Retail Strategy and Analytics

This notebook is my first pass at the chip transaction and customer data. The plan is to clean up the data, sanity check it, and then look for any useful patterns in who buys chips and how.

In [ ]:
# Import the libraries I need for this analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import xlrd
import re

In [ ]:
# Read in the two data files
customerdata = pd.read_csv('QVI_purchase_behaviour.csv')
transactiondata = pd.read_excel('QVI_transaction_data.xlsx')

## Exploratory Data Analysis

First I want to look at the data and check it's in good shape before doing any real analysis.

In [ ]:
# Keep a copy of the transaction data so I can always come back to the original
trans_df = transactiondata.copy()
trans_df.head()

The DATE column is stored as a number instead of an actual date, so I need to fix that first.

In [ ]:
# Convert DATE from an Excel-style integer into a real date
trans_df['DATE'] = pd.to_datetime(trans_df['DATE'], unit='D', origin='1899-12-30')
trans_df['DATE'].dtype

Next, I want to check that we're really only looking at chip purchases.

In [ ]:
# Look at all the different product names in the data
trans_df['PROD_NAME'].unique()

It looks like these are mostly chips, but I want to double check by counting how often each word shows up in the product name. To make that easier, I'll strip out the numbers and symbols first.

In [ ]:
# Remove the pack size (e.g. 175g) from the product name
prod_name = trans_df['PROD_NAME'].str.replace(r'[0-9]+[gG]', '', regex=True)

# Replace & with a space so flavours are split into separate words
prod_name = prod_name.str.replace('&', ' ', regex=False)

In [ ]:
# Split the cleaned up names into individual words and count how often each one appears
all_words = ' '.join(prod_name).split()
word_counts = pd.Series(all_words).value_counts()
word_counts

A few of the entries are actually salsa, not chips, so I'll remove those rows.

In [ ]:
# Remove salsa products - they aren't chips
trans_df = trans_df[trans_df['PROD_NAME'].str.contains('salsa', case=False) == False]
trans_df.shape

Now I'll look at some basic summary stats to check for outliers or missing values.

In [ ]:
trans_df.describe()

In [ ]:
# Check for any missing values anywhere in the data
trans_df.isnull().values.any()

The max PROD_QTY is 200, which looks unusually high. Let's look at that transaction more closely.

In [ ]:
trans_df[trans_df['PROD_QTY'] == 200]

The same loyalty card shows up twice, both with 200 packets. This could be a business account, so let's check if the same customer made any other, more normal purchases.

In [ ]:
trans_df[trans_df['LYLTY_CARD_NBR'] == 226000]

These are the only two purchases on this card, so it doesn't look like a typical retail customer. I'll remove these rows.

In [ ]:
trans_df = trans_df[trans_df['LYLTY_CARD_NBR'] != 226000]
trans_df.shape  # should be 2 rows fewer than before

In [ ]:
trans_df.describe()

That looks a lot more reasonable now. Next I'll check the number of transactions per day, in case any days are missing from the data.

In [ ]:
# Count how many transaction rows we have for each date
daily_counts = trans_df.groupby(trans_df['DATE'].dt.date).size()
daily_counts.shape

In [ ]:
trans_df.sort_values(by='DATE')

The dates run from 1 July 2018 to 30 June 2019, which should be 365 days, but the count above shows one fewer than that. Let's find out which day is missing.

In [ ]:
# Build the full list of dates for the year and see which ones are missing from our data
all_dates = pd.date_range(start='2018-07-01', end='2019-06-30')
dates_with_data = pd.to_datetime(daily_counts.index)

missing_dates = [d for d in all_dates if d not in dates_with_data]
missing_dates

The missing date is Christmas Day - a public holiday - so it makes sense that there were no sales that day.

Now I'll create a couple of extra features: the pack size and the brand name.

In [ ]:
# Pull the pack size (the number just before the g) out of the product name
trans_df['PACK_SIZE'] = trans_df['PROD_NAME'].str.extract(r'(\d+)').astype(float)
trans_df.sort_values(by='PACK_SIZE')

In [ ]:
# The smallest pack is 70g and the biggest is 380g, which looks reasonable.
# Plot a histogram to see how the pack sizes are distributed.
plt.hist(trans_df['PACK_SIZE'], weights=trans_df['PROD_QTY'])
plt.xlabel('Packet size (g)')
plt.ylabel('Quantity sold')
plt.title('Distribution of Pack Sizes Sold')
plt.show()

Now that the pack size looks fine, I can create the brand name using the first word of the product name.

In [ ]:
# The brand name is usually the first word in the product name
trans_df['BRAND_NAME'] = trans_df['PROD_NAME'].str.split().str[0]
trans_df['BRAND_NAME'].unique()

A few brands are showing up more than once under a shortened name. I'll fix these using a simple lookup table.

In [ ]:
# Map the shortened / duplicate brand names to one consistent name
brand_name_fixes = {
    'Infzns': 'Infuzions',
    'Red': 'Red Rock Deli',
    'RRD': 'Red Rock Deli',
    'Grain': 'Grain Waves',
    'GrnWves': 'Grain Waves',
    'Snbts': 'Sunbites',
    'Natural': 'Natural Chip Co',
    'NCC': 'Natural Chip Co',
    'WW': 'Woolworths',
    'Smith': 'Smiths',
    'Dorito': 'Doritos'
}

trans_df['BRAND_NAME'] = trans_df['BRAND_NAME'].replace(brand_name_fixes)
trans_df['BRAND_NAME'].unique()

The brand names look clean now, no duplicates left.

Now let's look at the customer data.

In [ ]:
cust_df = customerdata.copy()
cust_df.head()

In [ ]:
# Rename PREMIUM_CUSTOMER to something a bit easier to read
cust_df = cust_df.rename(columns={'PREMIUM_CUSTOMER': 'MEMBER_TYPE'})

In [ ]:
cust_df.describe()

In [ ]:
cust_df['MEMBER_TYPE'].unique()

In [ ]:
cust_df['LIFESTAGE'].unique()

The customer data looks fine too, so now I'll join it onto the transaction data.

In [ ]:
# Join the transaction and customer data together using the loyalty card number
full_df = pd.merge(trans_df, cust_df, on='LYLTY_CARD_NBR', how='left')
full_df = full_df.sort_values(by='DATE').reset_index(drop=True)
full_df.head()

In [ ]:
full_df.isnull().values.any()

In [ ]:
# Everything looks good, so I'll save this as a csv to use later
full_df.to_csv('QVI_fulldata.csv', index=False)

## Data analysis on customer segments

Now that the data is clean, I want to look for some interesting patterns in the chip market that could help with a business strategy. Some things I want to look at:
- Who spends the most on chips (total sales), by lifestage and how premium their purchasing behaviour is
- How many customers are in each segment
- How many chips are bought per customer, by segment
- The average chip price by customer segment

It would also be useful to get, from the data team:
- Each customer's total grocery spend, to see what share of it goes on chips
- Spending on other snacks (crackers, biscuits) to compare against chips
- The overall size of each customer segment, to compare against chip sales

In [ ]:
# Total sales by lifestage and member type
total_sales_cust = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['TOT_SALES'].sum().reset_index()
total_sales_cust = total_sales_cust.rename(columns={'TOT_SALES': 'sum_tot_sales'})
total_sales_cust.sort_values(by='sum_tot_sales', ascending=False)

In [ ]:
total_sales = full_df['TOT_SALES'].sum()

# Plot a breakdown of total sales by lifestage and member type
total_sales_breakdown = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['TOT_SALES'].sum().unstack('MEMBER_TYPE').fillna(0)
ax = total_sales_breakdown.plot(kind='barh', stacked=True, figsize=(15, 5))
ax.set_xlabel('Total sales ($)')
ax.set_title('Total Sales by Customer Lifestage and Affluence')
plt.show()

# The chart is a bit hard to read exact numbers off, so print the top segments as a
# percentage of total sales as well
top_segments = total_sales_cust.sort_values(by='sum_tot_sales', ascending=False).copy()
top_segments['pct_of_total'] = top_segments['sum_tot_sales'] / total_sales * 100
top_segments.head()

The biggest contributors to total sales are Older families - Budget, Young singles/couples - Mainstream, and Retirees - Mainstream. Let's check if that's just because there are more customers in those segments.

In [ ]:
# Check that every customer only appears once in the customer data
len(cust_df['LYLTY_CARD_NBR'].unique()) == cust_df.shape[0]

In [ ]:
# Check whether every customer actually bought chips at least once
len(cust_df['LYLTY_CARD_NBR'].unique()) == len(full_df['LYLTY_CARD_NBR'].unique())

In [ ]:
# Count the number of unique customers in each segment
sum_customers = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['LYLTY_CARD_NBR'].nunique().unstack('MEMBER_TYPE').fillna(0)
ax = sum_customers.plot(kind='barh', stacked=True, figsize=(15, 5))
ax.set_xlabel('No of customers')
ax.set_title('Number of Customers by Customer Lifestage and Affluence')
plt.show()

sum_customers

There are more Young singles/couples - Mainstream and Retirees - Mainstream customers, which explains why those segments have higher total sales. But that isn't the case for Older families - Budget, so something else must be driving their sales.

Let's look at the average number of chip packets bought per customer, by segment.

In [ ]:
# Average number of packets bought per customer, by segment
total_packets = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['PROD_QTY'].sum()
n_customers = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['LYLTY_CARD_NBR'].nunique()
avg_packets = (total_packets / n_customers).unstack('MEMBER_TYPE').fillna(0)

ax = avg_packets.plot.bar(figsize=(10, 5))
ax.set_ylabel('Avg no packets purchased')
ax.set_title('Chips Purchased by Customer Lifestage and Affluence')
plt.xticks(rotation=45)
plt.show()

Older families and young families buy more chips per customer than other groups. That helps explain why Older families - Budget contributes so much to total sales even though the segment isn't especially large.

Next, let's look at the average price paid per unit, by segment.

In [ ]:
full_df['UNIT_PRICE'] = full_df['TOT_SALES'] / full_df['PROD_QTY']

In [ ]:
avg_priceperunit = full_df.groupby(['LIFESTAGE', 'MEMBER_TYPE'])['UNIT_PRICE'].mean().unstack('MEMBER_TYPE').fillna(0)
ax = avg_priceperunit.plot.bar(figsize=(10, 5))
ax.set_ylabel('Avg unit price per transaction')
ax.set_title('Average Unit Price Per Transaction by Customer Lifestage and Affluence')
plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.xticks(rotation=45)
plt.show()

For young and midage singles/couples, the mainstream group seems willing to pay a bit more per packet than the budget or premium group. Combined with their higher total sales and customer numbers, this could suggest chips aren't the main snack choice for the non-mainstream groups - it would be worth getting more information on shopping habits to check this.

### Deep dive into specific customer segments

We've found a few interesting patterns already. Let's dig a bit deeper into one of the biggest segments - Mainstream young singles/couples - and see if they tend to prefer particular brands of chips.

In [ ]:
# Look at which brands young mainstream singles/couples buy the most of
young_mainstream = full_df[(full_df['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') & (full_df['MEMBER_TYPE'] == 'Mainstream')]

ax = young_mainstream['BRAND_NAME'].value_counts().sort_values().plot.barh(figsize=(10, 5))
ax.set_xlabel('No packets purchased')
ax.set_ylabel('Brands')
plt.show()

To get a general sense of the top brand in every segment (not just this one), I'll build a simple crosstab of segment vs brand.

In [ ]:
# Build a table showing how many packets each segment bought of each brand
temp = full_df.copy()
temp['SEGMENT'] = temp['LIFESTAGE'] + ' - ' + temp['MEMBER_TYPE']

segment_brand_counts = pd.crosstab(temp['SEGMENT'], temp['BRAND_NAME'])
segment_brand_counts.head()

In [ ]:
# Find the single most popular brand within each segment
top_brand_by_segment = segment_brand_counts.idxmax(axis=1)
top_brand_by_segment

Kettle comes out as the top brand for most segments, including Mainstream young singles/couples. Since that's true for almost everyone, it's not that useful for targeting this segment specifically. To find out what this segment prefers *more than other segments*, I'll use an affinity index - basically the ratio of how often this segment buys a brand compared to everyone else.

In [ ]:
# What share of this segment's purchases does each brand make up?
target_segment = young_mainstream['BRAND_NAME'].value_counts().rename_axis('BRANDS').reset_index(name='target')
target_segment['target'] = target_segment['target'] / young_mainstream['PROD_QTY'].sum()

# What share of everyone else's purchases does each brand make up?
not_young_mainstream = full_df[(full_df['LIFESTAGE'] != 'YOUNG SINGLES/COUPLES') & (full_df['MEMBER_TYPE'] != 'Mainstream')]
other = not_young_mainstream['BRAND_NAME'].value_counts().rename_axis('BRANDS').reset_index(name='other')
other['other'] = other['other'] / not_young_mainstream['PROD_QTY'].sum()

# Join the two together and calculate the affinity (how much more/less likely this segment is to buy each brand)
brand_proportions = pd.merge(target_segment, other, on='BRANDS')
brand_proportions['affinity'] = brand_proportions['target'] / brand_proportions['other']
brand_proportions.sort_values(by='affinity', ascending=False)

Mainstream young singles/couples are about 28% more likely to buy Tyrrells chips than other segments, but 50% less likely to buy Burger Rings.

Let's also check if this segment tends to prefer larger pack sizes.

In [ ]:
ax = young_mainstream['PACK_SIZE'].value_counts().sort_values().plot.barh()
ax.set_ylabel('Packet size (g)')
ax.set_xlabel('Packets purchased')
plt.show()

In [ ]:
# Also check which brands correspond to which pack sizes
brand_size = young_mainstream.groupby(['BRAND_NAME', 'PACK_SIZE'])['TOT_SALES'].sum().reset_index()
brand_size['label'] = brand_size['BRAND_NAME'] + ' - ' + brand_size['PACK_SIZE'].astype(int).astype(str) + 'g'
brand_size = brand_size.sort_values(by='TOT_SALES')

ax = brand_size.plot.barh(x='label', y='TOT_SALES', figsize=(10, 10), legend=False)
ax.set_ylabel('Brand and packet size')
ax.set_xlabel('Total sales ($)')
plt.show()

Most segments buy more of the 175g packets, which lines up with Kettle being the top brand overall since most Kettle chips come in 175g bags. Let's use the affinity index again to see if Mainstream young singles/couples prefer different pack sizes to everyone else.

In [ ]:
target_segment = young_mainstream['PACK_SIZE'].value_counts().rename_axis('SIZES').reset_index(name='target')
target_segment['target'] = target_segment['target'] / young_mainstream['PROD_QTY'].sum()

other = not_young_mainstream['PACK_SIZE'].value_counts().rename_axis('SIZES').reset_index(name='other')
other['other'] = other['other'] / not_young_mainstream['PROD_QTY'].sum()

brand_proportions = pd.merge(target_segment, other, on='SIZES')
brand_proportions['affinity'] = brand_proportions['target'] / brand_proportions['other']
brand_proportions.sort_values(by='affinity', ascending=False)

This segment is about 32% more likely to buy 270g bags than other segments, but 50% less likely to buy 220g bags. The 270g bags are Twisties, and the 220g bags are Burger Rings - which lines up with what we already saw when comparing brands.

## Summary of Insights

The three biggest contributors to total sales are:
1. Older families - Budget
2. Young singles/couples - Mainstream
3. Retirees - Mainstream

Mainstream young singles/couples and Mainstream retirees are the largest customer groups, which explains their high total sales. Population size isn't what's driving Older families - Budget though - older and young families in general just buy more chips per customer.

Mainstream young singles/couples also pay more per purchase on average than other young/midage singles and couples, and this difference is statistically significant. Looking closer at this segment, they buy Kettle chips the most (which is true of most segments), but compared to everyone else they're about 28% more likely to buy Tyrrells and about 50% less likely to buy Burger Rings. They also lean towards 270g packs (Twisties) over 220g packs (Burger Rings), which lines up with the brand preferences above.